# VibeML: Evaluation Traces

This notebook runs the VibeML agent through five trace scenarios and evaluates the results using MLflow. One trace runs the same scenario through two different LLMs to compare performance. Two traces test how the agent handles requests that fall outside its scope. A custom judge is used to score the agent's recommendations, and commentary is provided on what the evaluation showed.

## Setup

This pulls in the tools and agent logic built in the previous notebook and sets up an MLflow experiment to track all five traces.


In [0]:
import mlflow
import json
import pandas as pd

# Set up the MLflow experiment for this notebook
mlflow.set_experiment("/Shared/VibeML_Evaluation_Traces")

print("MLflow experiment set")

## Bringing in the Agent

This pulls in everything from the agent prototype notebook, including the tools, the Spotify connection, the LLM setup, and the datasets, so we can run the agent here without rebuilding it.

In [0]:
%run "./02_agent_prototype"

## Trace 1: Gym / High Energy (Dual LLM Comparison)

This trace runs the same user request through two different LLMs, databricks-gpt-oss-120b and databricks-meta-llama-3-3-70b-instruct, so we can compare how each one interprets the mood and picks songs.

In [0]:
user_request_1 = "I want fast afrobeat music for the gym"

with mlflow.start_run(run_name="trace1_gpt_oss_120b"):
    mlflow.log_param("llm_model", "databricks-gpt-oss-120b")
    mlflow.log_param("user_input", user_request_1)
    
    ranges_gpt = vibe_mapping(user_request_1, profile)
    explicit_genre = extract_explicit_genre(user_request_1)
    if explicit_genre:
        ranges_gpt["genre"] = explicit_genre
    
    results_gpt = search_songs(
        energy=(ranges_gpt["energy"]["min"], ranges_gpt["energy"]["max"]),
        valence=(ranges_gpt["valence"]["min"], ranges_gpt["valence"]["max"]),
        danceability=(ranges_gpt["danceability"]["min"], ranges_gpt["danceability"]["max"]),
        genre=ranges_gpt.get("genre"),
        limit=20,
        profile=profile,
        df_tracks=df_tracks,
        df_popularity=df_popularity
    )
    results_gpt = constraint_filter(results_gpt, explicit=False, min_popularity=30)
    results_gpt = deduplication(results_gpt, max_per_artist=1)
    
    top_songs_gpt = results_gpt.head(5)[["track_name", "artists", "track_genre"]].to_dict("records")
    
    mlflow.log_dict(ranges_gpt, "vibe_mapping_result.json")
    mlflow.log_dict(top_songs_gpt, "top_5_songs.json")
    mlflow.log_metric("songs_found", len(results_gpt))
    
    print(f"Model: databricks-gpt-oss-120b")
    print(f"Genre: {ranges_gpt.get('genre')}")
    print(f"Songs found: {len(results_gpt)}")
    for song in top_songs_gpt:
        print(f"  {song['track_name']} by {song['artists']}")

In [0]:
LLM_MODEL_2 = "databricks-meta-llama-3-3-70b-instruct"

with mlflow.start_run(run_name="trace1_llama_3_3_70b"):
    mlflow.log_param("llm_model", LLM_MODEL_2)
    mlflow.log_param("user_input", user_request_1)
    
    # Temporarily swap the model for vibe_mapping's LLM call
    original_model = LLM_MODEL
    LLM_MODEL = LLM_MODEL_2
    
    ranges_llama = vibe_mapping(user_request_1, profile)
    explicit_genre = extract_explicit_genre(user_request_1)
    if explicit_genre:
        ranges_llama["genre"] = explicit_genre
    
    LLM_MODEL = original_model  # restore original model
    
    results_llama = search_songs(
        energy=(ranges_llama["energy"]["min"], ranges_llama["energy"]["max"]),
        valence=(ranges_llama["valence"]["min"], ranges_llama["valence"]["max"]),
        danceability=(ranges_llama["danceability"]["min"], ranges_llama["danceability"]["max"]),
        genre=ranges_llama.get("genre"),
        limit=20,
        profile=profile,
        df_tracks=df_tracks,
        df_popularity=df_popularity
    )
    results_llama = constraint_filter(results_llama, explicit=False, min_popularity=30)
    results_llama = deduplication(results_llama, max_per_artist=1)
    
    top_songs_llama = results_llama.head(5)[["track_name", "artists", "track_genre"]].to_dict("records")
    
    mlflow.log_dict(ranges_llama, "vibe_mapping_result.json")
    mlflow.log_dict(top_songs_llama, "top_5_songs.json")
    mlflow.log_metric("songs_found", len(results_llama))
    
    print(f"Model: {LLM_MODEL_2}")
    print(f"Genre: {ranges_llama.get('genre')}")
    print(f"Songs found: {len(results_llama)}")
    for song in top_songs_llama:
        print(f"  {song['track_name']} by {song['artists']}")

### Comparison: GPT-OSS-120B vs Llama 3.3 70B

In [0]:
comparison = pd.DataFrame({
    "Model": ["databricks-gpt-oss-120b", "databricks-meta-llama-3-3-70b-instruct"],
    "Genre Detected": [ranges_gpt.get("genre"), ranges_llama.get("genre")],
    "Energy Range": [
        f"{ranges_gpt['energy']['min']}-{ranges_gpt['energy']['max']}",
        f"{ranges_llama['energy']['min']}-{ranges_llama['energy']['max']}"
    ],
    "Songs Found": [len(results_gpt), len(results_llama)],
    "Top Pick": [top_songs_gpt[0]["track_name"], top_songs_llama[0]["track_name"]]
})

comparison

### Commentary: Trace 1

Both models landed on the same genre and the same energy range for this request, which is a good sign that the vibe mapping logic is consistent across LLMs for a clear, unambiguous input like "fast afrobeat music for the gym." The difference shows up in the actual song picks. GPT-OSS-120B returned one more matching track than Llama 3.3 70B, likely due to small differences in how each model phrased its internal reasoning before settling on the same numeric ranges, which slightly changes which rows pass the filter at the edges. Neither model hallucinated a genre outside the dataset, and both correctly avoided non-afrobeat songs sneaking in. For this scenario, GPT-OSS-120B is the stronger pick simply because it surfaced more usable results from the same filters.

## Trace 2: Late Night Drive

This trace tests a more abstract, mood-based request rather than a direct genre request. The user describes a feeling rather than naming a genre, so this checks how well the agent translates context into audio features without an explicit genre override.

In [0]:
user_request_2 = "I'm going on a late night drive, give me something that matches that vibe"

with mlflow.start_run(run_name="trace2_late_night_drive"):
    mlflow.log_param("llm_model", LLM_MODEL)
    mlflow.log_param("user_input", user_request_2)
    
    ranges_2 = vibe_mapping(user_request_2, profile)
    explicit_genre_2 = extract_explicit_genre(user_request_2)
    if explicit_genre_2:
        ranges_2["genre"] = explicit_genre_2
    
    results_2 = search_songs(
        energy=(ranges_2["energy"]["min"], ranges_2["energy"]["max"]),
        valence=(ranges_2["valence"]["min"], ranges_2["valence"]["max"]),
        danceability=(ranges_2["danceability"]["min"], ranges_2["danceability"]["max"]),
        genre=ranges_2.get("genre"),
        limit=20,
        profile=profile,
        df_tracks=df_tracks,
        df_popularity=df_popularity
    )
    results_2 = constraint_filter(results_2, explicit=False, min_popularity=30)
    results_2 = deduplication(results_2, max_per_artist=1)
    
    top_songs_2 = results_2.head(5)[["track_name", "artists", "track_genre", "energy", "valence"]].to_dict("records")
    
    mlflow.log_dict(ranges_2, "vibe_mapping_result.json")
    mlflow.log_dict(top_songs_2, "top_5_songs.json")
    mlflow.log_metric("songs_found", len(results_2))
    
    print(f"Genre: {ranges_2.get('genre')}")
    print(f"Energy range: {ranges_2['energy']['min']}-{ranges_2['energy']['max']}")
    print(f"Valence range: {ranges_2['valence']['min']}-{ranges_2['valence']['max']}")
    print(f"Songs found: {len(results_2)}")
    for song in top_songs_2:
        print(f"  {song['track_name']} by {song['artists']} (energy: {song['energy']}, valence: {song['valence']})")

### Commentary: Trace 2

This trace had no explicit genre mentioned, so the agent had to infer one purely from the mood description. It landed on deep-house, which fits a late night drive reasonably well since the energy is moderate and the valence leans lower, meaning the songs feel more atmospheric than upbeat. This shows the vibe mapping tool is not just pattern matching on keywords, it is actually reasoning about what kind of music fits an unstated genre based on context. Songs found dropped slightly compared to Trace 1, which makes sense since deep-house is a smaller genre in the dataset compared to afrobeat, but 18 results is still a healthy pool to pick tester songs from.

## Trace 3: Studying / No Lyrics

This trace tests a functional request rather than a mood-based one. The user wants something specific for focus, low energy, no lyrics, which checks whether the agent correctly prioritizes low speechiness and high instrumentalness instead of just defaulting to a generic "calm" genre.

In [0]:
user_request_3 = "I need to focus, I'm studying for an exam. Something calm with no lyrics please"

with mlflow.start_run(run_name="trace3_studying_no_lyrics"):
    mlflow.log_param("llm_model", LLM_MODEL)
    mlflow.log_param("user_input", user_request_3)
    
    ranges_3 = vibe_mapping(user_request_3, profile)
    explicit_genre_3 = extract_explicit_genre(user_request_3)
    if explicit_genre_3:
        ranges_3["genre"] = explicit_genre_3
    
    results_3 = search_songs(
        energy=(ranges_3["energy"]["min"], ranges_3["energy"]["max"]),
        valence=(ranges_3["valence"]["min"], ranges_3["valence"]["max"]),
        danceability=(ranges_3["danceability"]["min"], ranges_3["danceability"]["max"]),
        speechiness=(ranges_3["speechiness"]["min"], ranges_3["speechiness"]["max"]),
        instrumentalness=(ranges_3["instrumentalness"]["min"], ranges_3["instrumentalness"]["max"]),
        genre=ranges_3.get("genre"),
        limit=20,
        profile=profile,
        df_tracks=df_tracks,
        df_popularity=df_popularity
    )
    results_3 = constraint_filter(results_3, explicit=False, min_popularity=30)
    results_3 = deduplication(results_3, max_per_artist=1)
    
    top_songs_3 = results_3.head(5)[["track_name", "artists", "track_genre", "speechiness", "instrumentalness"]].to_dict("records")
    
    mlflow.log_dict(ranges_3, "vibe_mapping_result.json")
    mlflow.log_dict(top_songs_3, "top_5_songs.json")
    mlflow.log_metric("songs_found", len(results_3))
    
    print(f"Genre: {ranges_3.get('genre')}")
    print(f"Speechiness range: {ranges_3['speechiness']['min']}-{ranges_3['speechiness']['max']}")
    print(f"Instrumentalness range: {ranges_3['instrumentalness']['min']}-{ranges_3['instrumentalness']['max']}")
    print(f"Songs found: {len(results_3)}")
    for song in top_songs_3:
        print(f"  {song['track_name']} by {song['artists']} (speechiness: {song['speechiness']}, instrumentalness: {song['instrumentalness']})")

### Commentary: Trace 3

This trace exposed a real limitation in the song search logic. The agent correctly identified the genre as "study" and set the right direction for speechiness and instrumentalness, low speech, high instrumental, but combining that genre filter with all four numeric ranges at once returned zero results, even after the smart fallback widened the ranges three times. The fallback logic worked as intended in the sense that it never returned an empty playlist to the user, it fell back to genre-only filtering once widening failed. The resulting songs are still a good fit by name and by their actual speechiness and instrumentalness values, so the end result is fine, but it shows that stacking a narrow genre like "study" with strict audio feature ranges is a combination the dataset can't always support. A possible improvement would be loosening the genre constraint before the audio features when the dataset is too sparse for a given genre.

## Trace 4: Graceful Rejection 1 — Podcast Request

This trace tests whether the agent correctly recognizes a request that falls outside its scope and declines it cleanly instead of trying to force a music-based answer or hallucinating a response.

In [0]:
user_request_4 = "Can you recommend a good podcast to listen to on my commute?"

with mlflow.start_run(run_name="trace4_rejection_podcast"):
    mlflow.log_param("llm_model", LLM_MODEL)
    mlflow.log_param("user_input", user_request_4)
    
    rejection_prompt = f"""You are VibeML, a personalized Spotify playlist agent. You only help with music playlist requests.
    
    A user just said: "{user_request_4}"
    
    Respond naturally, explaining that this falls outside what you can help with since you are scoped to music playlists, 
    and then redirect them back to building a playlist."""
    
    rejection_response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": rejection_prompt}]
    )
    
    content = rejection_response.choices[0].message.content
    if isinstance(content, list):
        rejection_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        rejection_text = content
    
    mlflow.log_text(rejection_text, "agent_response.txt")
    
    print(f"User: {user_request_4}")
    print(f"\nVibeML: {rejection_text}")

### Commentary: Trace 4

The agent handled this rejection well. It did not try to answer the podcast question or pretend it had podcast recommendations, it clearly stated that podcasts are outside its scope and immediately pivoted back to offering a playlist instead. This is the behavior we want, the agent stays in character as a music playlist assistant and turns a rejected request into an opportunity to keep the conversation productive rather than just shutting the user down.

## Trace 5: Graceful Rejection 2 — Financial Advice Request

This trace tests a different kind of out-of-scope request, one that is not music-adjacent at all. This checks whether the agent holds its boundaries consistently across different types of irrelevant requests, not just ones that are loosely related to audio content like the podcast example.

In [0]:
user_request_5 = "Hey, what stocks should I invest in right now?"

with mlflow.start_run(run_name="trace5_rejection_stocks"):
    mlflow.log_param("llm_model", LLM_MODEL)
    mlflow.log_param("user_input", user_request_5)
    
    rejection_prompt_2 = f"""You are VibeML, a personalized Spotify playlist agent. You only help with music playlist requests.
    
    A user just said: "{user_request_5}"
    
    Respond naturally, explaining that this falls outside what you can help with since you are scoped to music playlists, 
    and then redirect them back to building a playlist."""
    
    rejection_response_2 = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": rejection_prompt_2}]
    )
    
    content = rejection_response_2.choices[0].message.content
    if isinstance(content, list):
        rejection_text_2 = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        rejection_text_2 = content
    
    mlflow.log_text(rejection_text_2, "agent_response.txt")
    
    print(f"User: {user_request_5}")
    print(f"\nVibeML: {rejection_text_2}")

### Commentary: Trace 5

This trace confirms the agent holds its scope consistently even for a request that has nothing to do with audio or media at all, unlike the podcast example which was at least loosely adjacent. The agent did not attempt to answer the financial question or make a disclaimer about not being a financial advisor in a roundabout way, it simply declined clearly and redirected to its actual purpose. Combined with Trace 4, this shows the rejection behavior is not narrowly tuned to one specific kind of out-of-scope request, it generalizes across different categories of irrelevant input.

## Custom Judge: Vibe Match Quality

This judge checks whether the songs the agent recommended actually fit what the user asked for. It looks at the user's original request and the resulting song list, then rates how well the recommendations match the requested vibe.

In [0]:
def vibe_match_judge(user_request, recommended_songs, genre_detected):
    """
    LLM-as-judge that scores how well a set of recommended songs matches
    the user's original request.
    
    Args:
        user_request (str): The original user input
        recommended_songs (list): List of song/artist dicts that were recommended
        genre_detected (str): The genre the vibe mapping tool landed on
    Returns:
        dict: Score (1-5), reasoning, and pass/fail flag
    """
    songs_text = "\n".join([f"- {s['track_name']} by {s['artists']}" for s in recommended_songs])
    
    judge_prompt = f"""You are evaluating whether a music recommendation agent did a good job.

User request: "{user_request}"
Genre the agent detected: {genre_detected}
Songs recommended:
{songs_text}

Rate how well these songs actually match what the user asked for, on a scale of 1 to 5.
1 means the songs do not match the request at all.
5 means the songs are a strong match for the request.

Return ONLY a JSON object in this exact format, no other text:
{{"score": 4, "reasoning": "short explanation here", "pass": true}}

A score of 3 or higher should have "pass": true. Below 3 should have "pass": false."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": judge_prompt}]
    )

    content = response.choices[0].message.content
    if isinstance(content, list):
        raw = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        raw = content

    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

# Quick test on Trace 1 results
test_judge_result = vibe_match_judge(user_request_1, top_songs_gpt, ranges_gpt.get("genre"))
print(json.dumps(test_judge_result, indent=2))

### Running the Judge Across All 5 Traces

In [0]:
judge_results = []

with mlflow.start_run(run_name="judge_evaluation_all_traces"):
    
    # Trace 1a - GPT-OSS
    r1a = vibe_match_judge(user_request_1, top_songs_gpt, ranges_gpt.get("genre"))
    judge_results.append({"trace": "Trace 1 (GPT-OSS-120B)", "request": user_request_1, **r1a})
    
    # Trace 1b - Llama
    r1b = vibe_match_judge(user_request_1, top_songs_llama, ranges_llama.get("genre"))
    judge_results.append({"trace": "Trace 1 (Llama 3.3 70B)", "request": user_request_1, **r1b})
    
    # Trace 2
    r2 = vibe_match_judge(user_request_2, top_songs_2, ranges_2.get("genre"))
    judge_results.append({"trace": "Trace 2", "request": user_request_2, **r2})
    
    # Trace 3
    r3 = vibe_match_judge(user_request_3, top_songs_3, ranges_3.get("genre"))
    judge_results.append({"trace": "Trace 3", "request": user_request_3, **r3})
    
    judge_df = pd.DataFrame(judge_results)
    
    mlflow.log_dict(judge_results, "judge_results_all_traces.json")
    mlflow.log_metric("average_score", judge_df["score"].mean())
    mlflow.log_metric("pass_rate", judge_df["pass"].mean())

judge_df

### Commentary: Judge Evaluation Results

The judge results show a clear difference between the two LLMs on Trace 1, the same gym/afrobeat request. GPT-OSS-120B scored a 2 and failed, while Llama 3.3 70B scored a 3 and passed, even though both models used the same downstream search and filtering logic. This means the difference came entirely from how each model interpreted the user's request inside vibe_mapping and which audio feature ranges it settled on, which then changed which songs got pulled from the dataset. For this scenario, Llama 3.3 70B produced a recommendation set the judge considered a better fit, despite GPT-OSS-120B finding one extra song earlier in the manual review. This is a useful finding because it shows that "more results" does not necessarily mean "better results," the judge is catching a quality gap that a simple count would have missed.

Trace 2 and Trace 3 both scored well, a 4 each, which lines up with what was found earlier, both produced songs that genuinely matched the requested mood, even though Trace 3 needed the fallback widening logic to get there. The lower score on Trace 1 versus the higher scores on Traces 2 and 3 also suggests the agent does better on mood-based, inferred-genre requests than on explicit genre requests where the dataset's genre tagging doesn't fully match listener expectations of that genre.

## Human-in-the-Loop Evaluation

The LLM judge gives us a fast, scalable way to score every trace, but it shouldn't be the only check. Here, we review the same traces the judge scored and give our own rating, then we compare where the judge agrees or disagrees. This is the same idea where a domain expert labels a batch of traces and that feedback is used to catch cases where the judge's criteria drift from what people actually expect.

In [0]:
# Human review of the 4 scored traces
human_labels = [
    {
        "trace": "Trace 1 (GPT-OSS-120B)",
        "human_score": 2,
        "human_reasoning": "Agree with the judge. Most of these don't actually sound like fast afrobeat gym music."
    },
    {
        "trace": "Trace 1 (Llama 3.3 70B)",
        "human_score": 2,
        "human_reasoning": "I'd actually score this lower than the judge did. Soul Makossa is a classic but it isn't fast or gym-energy, so I don't fully agree these are a strong fit either."
    },
    {
        "trace": "Trace 2",
        "human_score": 4,
        "human_reasoning": "Agree with the judge. These deep-house picks genuinely fit a late night drive."
    },
    {
        "trace": "Trace 3",
        "human_score": 4,
        "human_reasoning": "Agree with the judge. These are calm, mostly instrumental, and would work well for studying."
    }
]

human_df = pd.DataFrame(human_labels)

# Merge with judge scores for side by side comparison
comparison_df = judge_df[["trace", "score", "pass"]].merge(
    human_df, on="trace"
).rename(columns={"score": "judge_score"})

comparison_df["agreement"] = comparison_df["judge_score"] == comparison_df["human_score"]

pd.set_option("display.max_colwidth", None)
comparison_df[["trace", "judge_score", "human_score", "agreement", "human_reasoning"]]

### Commentary: Human-in-the-Loop Review

The human review agreed with the judge on 3 out of 4 traces. The one disagreement is meaningful: on Trace 1 run through Llama 3.3 70B, the judge scored it a 3 (a pass), but the human review scored it a 2. The judge's reasoning leaned on the fact that the tracks were "Afro-inspired," while the human reasoning was stricter about what "fast" and "gym energy" actually mean in practice, a song like Soul Makossa is a recognizable Afrobeat classic, but it does not have the tempo or intensity someone would actually want during a workout. This is a small but real example of criteria drift, the judge's definition of "good enough" was looser than what an actual user would expect for this specific request. It also reinforces why a judge should not be the only check, an LLM judge can rate songs as a reasonable genre match without picking up on whether the energy actually fits the stated activity. If this were taken further, this disagreement is exactly the kind of labeled example that could be used to align the judge going forward, the same approach used in Module 6, where expert labels are compared against judge output to tighten the judge's criteria.

## Overall Agent Performance Summary

This section ties together what the five traces and the evaluation showed about how the agent performs overall, and compares the two LLMs directly.

### Final Commentary

Across the five traces, the agent performed well on mood-based requests and reasonably on explicit genre requests, with one clear weak spot worth calling out.

**Genre detection and search behavior:** The agent correctly detected genre in every trace, whether the user named one directly (Trace 1) or only described a mood (Trace 2 and Trace 3). The smart fallback logic in the song search tool also did its job in Trace 3, when the combination of genre plus strict audio feature ranges returned zero results, the agent widened the ranges automatically rather than failing outright. That said, Trace 3 also showed the limit of this approach, widening still wasn't enough and the tool had to fall back to genre-only filtering, which is a gap worth fixing in a future version.

**LLM comparison:** Trace 1 is the clearest direct comparison since the same input was run through both databricks-gpt-oss-120b and databricks-meta-llama-3-3-70b-instruct. GPT-OSS-120B returned one more raw result, but the judge actually scored it lower (2 versus 3) than Llama 3.3 70B on the same request. The team's own review pushed back on the judge a little here, scoring Llama's output a 2 as well, since one of its top picks (Soul Makossa) doesn't really fit a fast gym vibe despite being a well-known Afrobeat track. Taken together, this suggests GPT-OSS-120B and Llama 3.3 70B perform similarly on interpreting straightforward audio feature ranges, but neither model is reliably translating "fast" and "gym energy" into picks that a real listener would consider on-vibe when the dataset's genre tagging is loose. Llama edged out GPT-OSS-120B on this particular request only because the judge weighted genre match over tempo and energy fit.

**Where the agent struggled:** The weakest result was Trace 1, the explicit genre request. This came down to the dataset itself, the afrobeat tag in the Kaggle dataset includes tracks that don't match what most listeners associate with the genre's sound, so even a working genre filter pulls in songs that miss the mark. The two mood-based requests, the late night drive and the studying session, scored much higher (4 out of 5 on both) because they relied more on audio feature matching than on the genre label being accurate.

**Where the agent succeeded:** The two graceful rejection traces (Trace 4 and Trace 5) were clean across the board, the agent consistently recognized out-of-scope requests, whether music-adjacent (podcasts) or completely unrelated (stock advice), and redirected the user back to playlist building without ever trying to fake an answer.

**Overall takeaway:** The agent is solid at staying in scope and at matching mood-based requests to audio features, but its accuracy on explicit genre requests is only as good as the genre tags in the underlying dataset. The human-in-the-loop review also showed the judge isn't perfectly calibrated either, it can rate something as a pass when a real listener would call it a near-miss, which is exactly the kind of gap that human review is meant to catch.